In [ ]:
import pandas 

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson",
    "countries.geojson"
)

In [ ]:
"""
FIFA Men's World Cup Winners Map (1930-2024)
Author: Muhammad Mehdi
Data Source: https://en.wikipedia.org/wiki/FIFA_World_Cup
"""

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# -----------------------------
# 1. Data (from Wikipedia - FIFA World Cup results, 1930-2022)
# -----------------------------
# Number of titles won by each country
champions = {
    "Brazil": 5,
    "Germany": 4,
    "Italy": 4,
    "Argentina": 3,
    "France": 2,
    "Uruguay": 2,
    "England": 1,
    "Spain": 1,
}

# Countries that have ever qualified/participated (sample - extend as needed)
participants = [
    "United States of America", "Mexico", "Canada", "Costa Rica", "Honduras",
    "Jamaica", "Panama", "Trinidad and Tobago", "Cuba", "Haiti",
    "Brazil", "Argentina", "Uruguay", "Chile", "Colombia", "Peru",
    "Ecuador", "Bolivia", "Paraguay", "Venezuela",
    "Spain", "Portugal", "France", "Germany", "Italy", "England",
    "Netherlands", "Belgium", "Switzerland", "Austria", "Sweden",
    "Denmark", "Norway", "Poland", "Romania", "Czechia", "Slovakia",
    "Croatia", "Serbia", "Bosnia and Herz.", "Hungary", "Bulgaria",
    "Greece", "Russia", "Ukraine", "Scotland", "Wales", "Iceland",
    "Morocco", "Tunisia", "Algeria", "Egypt", "Nigeria", "Ghana",
    "Cameroon", "Senegal", "Ivory Coast", "South Africa", "Togo",
    "Saudi Arabia", "Iran", "South Korea", "Japan", "Australia",
    "China", "United Arab Emirates", "Israel", "New Zealand",
    "Indonesia", "North Korea", "Qatar", "Kuwait", "Iraq"
]

# -----------------------------
# 2. Load world map geometry
# -----------------------------
world = gpd.read_file("countries.geojson")

# -----------------------------
# 3. Classify countries
# -----------------------------
world["status"] = "Other"
world.loc[world["NAME"].isin(participants), "status"] = "Participants"
world.loc[world["NAME"].isin(champions.keys()), "status"] = "Champions"

color_map = {
    "Other": "#d9d9d9",
    "Participants": "#4a7c3f",
    "Champions": "#f4c542",
}
world["color"] = world["status"].map(color_map)

# -----------------------------
# 4. Plot the map
# -----------------------------
fig, ax = plt.subplots(figsize=(16, 9))
fig.patch.set_facecolor("white")

world.plot(color=world["color"], ax=ax, edgecolor="white", linewidth=0.4)

ax.set_axis_off()
ax.set_title(
    "FIFA Men's World Cup Winners\n1930 - 2022",
    fontsize=22, fontweight="bold", pad=20
)

# -----------------------------
# 5. Add title-count labels on champion countries
# -----------------------------
# Manual offsets (dx, dy) to avoid label overlap in crowded regions (Europe)
offsets = {
    "Germany": (5, 6),
    "Italy": (5, -3),
    "France": (-8, 3),
    "Spain": (-10, -3),
    "England": (-3, 8),
    "Argentina": (-6, -3),
    "Uruguay": (4, -2),
    "Brazil": (0, 0),
}

for country, titles in champions.items():
    row = world[world["NAME"] == country]
    if not row.empty:
        centroid = row.geometry.centroid.iloc[0]
        dx, dy = offsets.get(country, (0, 0))
        ax.annotate(
            f"{country}\n{titles} Title{'s' if titles > 1 else ''}",
            xy=(centroid.x, centroid.y),
            xytext=(centroid.x + dx, centroid.y + dy),
            ha="center", va="center",
            fontsize=8, fontweight="bold", color="black",
            arrowprops=dict(arrowstyle="-", color="black", lw=0.6),
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.85)
        )

# -----------------------------
# 6. Legend
# -----------------------------
legend_elements = [
    Patch(facecolor="#f4c542", edgecolor="black", label="Champions"),
    Patch(facecolor="#4a7c3f", edgecolor="black", label="Participants"),
    Patch(facecolor="#d9d9d9", edgecolor="black", label="Other Countries"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=10, frameon=True)

# -----------------------------
# 7. Footer: data source + author credit
# -----------------------------
fig.text(
    0.5, 0.02,
    "Data Source: https://en.wikipedia.org/wiki/FIFA_World_Cup   |   © 2026 Muhammad Mehdi",
    ha="center", fontsize=9, color="gray"
)

plt.tight_layout()
#plt.savefig("worldcup_map.png", dpi=200, bbox_inches="tight")
plt.show()